In [1]:
!pip uninstall -y transformers peft accelerate trl huggingface_hub
!pip install -q unsloth
!pip install -q datasets sentence-transformers faiss-cpu qiskit qiskit-machine-learning

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: huggingface_hub 1.10.1
Uninstalling huggingface_hub-1.10.1:
  Successfully uninstalled huggingface_hub-1.10.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 515.2 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 MB 21.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 25.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 671.5/671.5 kB 45.7 MB/s eta 0

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("gvaldenebro/cancer-q-and-a-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/gvaldenebro/cancer-q-and-a-dataset


In [3]:
import pandas as pd
import glob

base_path = "/kaggle/input"

csv_files = glob.glob(base_path + "/**/*.csv", recursive=True)

print("Found files:", len(csv_files))

dfs = []

for file in csv_files:
    try:
        df = pd.read_csv(file)

        if "Question" in df.columns and "Answer" in df.columns:
            df = df[["Question", "Answer"]]
            dfs.append(df)
            print(f"Loaded: {file} | {len(df)} rows")

    except Exception as e:
        print(file, e)

df = pd.concat(dfs, ignore_index=True)

df = df.dropna()

df = df.drop_duplicates()

print("Final Shape:", df.shape)

df.head()

Found files: 10
Loaded: /kaggle/input/datasets/gvaldenebro/cancer-q-and-a-dataset/growth_hormone_receptorQA.csv | 5430 rows
Loaded: /kaggle/input/datasets/gvaldenebro/cancer-q-and-a-dataset/MedicalQuestionAnswering.csv | 16406 rows
Loaded: /kaggle/input/datasets/gvaldenebro/cancer-q-and-a-dataset/Disease_Control_and_PreventionQA.csv | 270 rows
Loaded: /kaggle/input/datasets/gvaldenebro/cancer-q-and-a-dataset/Genetic_and_Rare_DiseasesQA.csv | 5388 rows
Loaded: /kaggle/input/datasets/gvaldenebro/cancer-q-and-a-dataset/Diabetes_and_Digestive_and_Kidney_DiseasesQA.csv | 1192 rows
Loaded: /kaggle/input/datasets/gvaldenebro/cancer-q-and-a-dataset/CancerQA.csv | 729 rows
Loaded: /kaggle/input/datasets/gvaldenebro/cancer-q-and-a-dataset/Neurological_Disorders_and_StrokeQA.csv | 1088 rows
Loaded: /kaggle/input/datasets/gvaldenebro/cancer-q-and-a-dataset/Heart_Lung_and_BloodQA.csv | 559 rows
Loaded: /kaggle/input/datasets/gvaldenebro/cancer-q-and-a-dataset/SeniorHealthQA.csv | 769 rows
Loaded: /

,Question,Answer
0,What is (are) keratoderma with woolly hair ?,Keratoderma with woolly hair is a group of rel...
1,How many people are affected by keratoderma wi...,Keratoderma with woolly hair is rare; its prev...
2,What are the genetic changes related to kerato...,"Mutations in the JUP, DSP, DSC2, and KANK2 gen..."
3,Is keratoderma with woolly hair inherited ?,Most cases of keratoderma with woolly hair hav...
4,What are the treatments for keratoderma with w...,These resources address the diagnosis or manag...


In [4]:
df["Question"] = (
    df["Question"]
    .astype(str)
    .str.strip()
)

df["Answer"] = (
    df["Answer"]
    .astype(str)
    .str.strip()
)

df = df[
    (df["Question"].str.len() > 10) &
    (df["Answer"].str.len() > 20)
]

df = df.drop_duplicates(
    subset=["Question"]
)

print(df.shape)

(14977, 2)


In [5]:
df["text"] = df.apply(
    lambda x:
    f"""<|im_start|>user
{x['Question']}
<|im_end|>

<|im_start|>assistant
{x['Answer']}
<|im_end|>""",
    axis=1
)

df = df[["text"]]

In [6]:
print(df.shape)
df.head(3)

(14977, 1)


,text
0,<|im_start|>user\nWhat is (are) keratoderma wi...
1,<|im_start|>user\nHow many people are affected...
2,<|im_start|>user\nWhat are the genetic changes...


In [7]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

print(dataset)

Dataset({
    features: ['text', '__index_level_0__'],
    num_rows: 14977
})


In [8]:
dataset = dataset.train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['text', '__index_level_0__'],
    num_rows: 13479
})
Dataset({
    features: ['text', '__index_level_0__'],
    num_rows: 1498
})


In [9]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-3B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [10]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

Unsloth 2026.6.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [12]:
from transformers import TrainingArguments
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=2048,

    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=50,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        optim="adamw_torch",
        output_dir="outputs",
        report_to="none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/13479 [00:00<?, ? examples/s]

In [13]:
model.save_pretrained("medical_qwen")
tokenizer.save_pretrained("medical_qwen")

Unsloth: Restored added_tokens_decoder metadata in medical_qwen/tokenizer_config.json.


('medical_qwen/tokenizer_config.json',
 'medical_qwen/chat_template.jinja',
 'medical_qwen/tokenizer.json')

In [14]:
question = "What are the symptoms of lung cancer?"

prompt = f"""
### Question:
{question}

### Answer:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.2,
)

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12


### Question:
What are the symptoms of lung cancer?

### Answer:
Lung cancer can cause a variety of symptoms, including:

- Persistent coughing that doesn't go away
- Coughing up blood or bloody sputum
- Shortness of breath
- Chest pain
- Unexplained weight loss
- Fatigue
- Nausea and vomiting
- Difficulty swallowing
- Hoarse voice
- Frequent infections like pneumonia or bronchitis

It's important to note that these symptoms can also be caused by other conditions, so if you experience any of them, it's important to see a doctor for a proper diagnosis.


In [15]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [21]:
import pandas as pd
import glob

csv_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)

dfs = []

for file in csv_files:
    try:
        temp = pd.read_csv(file)

        if "Question" in temp.columns and "Answer" in temp.columns:
            temp = temp[["Question", "Answer"]]
            dfs.append(temp)

    except:
        pass

rag_df = pd.concat(dfs, ignore_index=True)

rag_df = rag_df.dropna()

print(rag_df.shape)

rag_df.head()

(32812, 2)


,Question,Answer
0,What is (are) keratoderma with woolly hair ?,Keratoderma with woolly hair is a group of rel...
1,How many people are affected by keratoderma wi...,Keratoderma with woolly hair is rare; its prev...
2,What are the genetic changes related to kerato...,"Mutations in the JUP, DSP, DSC2, and KANK2 gen..."
3,Is keratoderma with woolly hair inherited ?,Most cases of keratoderma with woolly hair hav...
4,What are the treatments for keratoderma with w...,These resources address the diagnosis or manag...


In [22]:
import pandas as pd
import glob

csv_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)

dfs = []

for file in csv_files:
    try:
        temp = pd.read_csv(file)

        if "Question" in temp.columns and "Answer" in temp.columns:
            temp = temp[["Question", "Answer"]]
            dfs.append(temp)

    except:
        pass

rag_df = pd.concat(dfs, ignore_index=True)

rag_df = rag_df.dropna()

print(rag_df.shape)

rag_df.head()

(32812, 2)


,Question,Answer
0,What is (are) keratoderma with woolly hair ?,Keratoderma with woolly hair is a group of rel...
1,How many people are affected by keratoderma wi...,Keratoderma with woolly hair is rare; its prev...
2,What are the genetic changes related to kerato...,"Mutations in the JUP, DSP, DSC2, and KANK2 gen..."
3,Is keratoderma with woolly hair inherited ?,Most cases of keratoderma with woolly hair hav...
4,What are the treatments for keratoderma with w...,These resources address the diagnosis or manag...


In [23]:
embeddings = embedder.encode(
    rag_df["Question"].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/1026 [00:00<?, ?it/s]

In [24]:
print(rag_df.columns)
print(rag_df.shape)

Index(['Question', 'Answer'], dtype='object')
(32812, 2)


In [26]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(
    "BAAI/bge-m3"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [27]:
embeddings = embedder.encode(
    rag_df["Question"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches:   0%|          | 0/1026 [00:00<?, ?it/s]

In [28]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

faiss.normalize_L2(embeddings)

index.add(
    embeddings.astype("float32")
)

print(index.ntotal)

32812


In [29]:
def retrieve_docs(
    query,
    top_k=50
):

    q_emb = embedder.encode(
        [query],
        convert_to_numpy=True
    )

    faiss.normalize_L2(q_emb)

    D, I = index.search(
        q_emb.astype("float32"),
        top_k
    )

    return rag_df.iloc[I[0]].copy()

In [30]:
def retrieve_docs(
    query,
    top_k=50
):

    q_emb = embedder.encode(
        [query],
        convert_to_numpy=True
    )

    faiss.normalize_L2(q_emb)

    D, I = index.search(
        q_emb.astype("float32"),
        top_k
    )

    return rag_df.iloc[I[0]].copy()

In [31]:
docs = retrieve_docs(
    "what are symptoms of lung cancer"
)

docs.head()

,Question,Answer
31511,What are the symptoms of Lung Cancer ?,The possible signs of lung cancer are: - a cou...
31498,What are the symptoms of Lung Cancer ?,Common Signs and Symptoms When lung cancer fir...
11309,What are the symptoms of Lung Cancer ?,The possible signs of lung cancer are: - a cou...
11296,What are the symptoms of Lung Cancer ?,Common Signs and Symptoms When lung cancer fir...
23327,What are the symptoms of Lung adenocarcinoma ?,What are the signs and symptoms of Lung adenoc...


In [32]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [33]:
def rerank_docs(
    query,
    docs,
    top_k=10
):

    pairs = [
        (query, q)
        for q in docs["Question"]
    ]

    scores = reranker.predict(
        pairs
    )

    docs = docs.copy()

    docs["cross_score"] = scores

    docs = docs.sort_values(
        "cross_score",
        ascending=False
    )

    return docs.head(top_k)

In [36]:
!pip install qiskit qiskit-machine-learning

In [37]:
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel

In [38]:
feature_map = ZZFeatureMap(
    feature_dimension=2,
    reps=2
)

quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map
)

/tmp/ipykernel_58/1792720035.py:1: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(


In [39]:
import numpy as np

def quantum_rerank(
    docs,
    top_k=3
):

    scores = docs["cross_score"].values

    scores = np.array(scores)

    scores = scores.reshape(-1,1)

    scores = scores[:10]

    second_feature = np.zeros_like(scores)

    X = np.hstack(
        [scores, second_feature]
    )

    kernel_matrix = quantum_kernel.evaluate(X)

    quantum_scores = kernel_matrix.mean(
        axis=1
    )

    docs = docs.iloc[:len(quantum_scores)].copy()

    docs["quantum_score"] = quantum_scores

    docs = docs.sort_values(
        "quantum_score",
        ascending=False
    )

    return docs.head(top_k)

In [40]:
query = "what are symptoms of lung cancer"

docs = retrieve_docs(
    query,
    50
)

docs = rerank_docs(
    query,
    docs,
    10
)

docs = quantum_rerank(
    docs,
    3
)

docs

,Question,Answer,cross_score,quantum_score
11309,What are the symptoms of Lung Cancer ?,The possible signs of lung cancer are: - a cou...,4.375861,0.644134
11296,What are the symptoms of Lung Cancer ?,Common Signs and Symptoms When lung cancer fir...,4.375861,0.644134
31511,What are the symptoms of Lung Cancer ?,The possible signs of lung cancer are: - a cou...,4.375861,0.644134


In [41]:
context = "\n\n".join(
    docs["Answer"].tolist()
)

print(context[:2000])

The possible signs of lung cancer are: - a cough that doesn't go away and gets worse over time  - constant chest pain  - coughing up blood  - shortness of breath, wheezing, or hoarseness  - repeated problems with pneumonia or bronchitis  - swelling of the neck and face  - loss of appetite or weight loss  - fatigue. a cough that doesn't go away and gets worse over time constant chest pain coughing up blood shortness of breath, wheezing, or hoarseness repeated problems with pneumonia or bronchitis swelling of the neck and face loss of appetite or weight loss fatigue.

Common Signs and Symptoms When lung cancer first develops, there may be no symptoms at all. But if the cancer grows, it can cause changes that people should watch for. Common signs and symptoms of lung cancer include: - a cough that doesn't go away and gets worse over time  - constant chest pain  - coughing up blood  - shortness of breath, wheezing, or hoarseness  - repeated problems with pneumonia or bronchitis  - swelling

In [42]:
prompt = f"""
You are a medical assistant.

Answer ONLY using the supplied context.

If the answer is not present in the context,
say:

'I could not find sufficient evidence.'

Context:

{context}

Question:

{query}

Answer:
"""

In [43]:
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.1,
    do_sample=False,
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



You are a medical assistant.

Answer ONLY using the supplied context.

If the answer is not present in the context,
say:

'I could not find sufficient evidence.'

Context:

The possible signs of lung cancer are: - a cough that doesn't go away and gets worse over time  - constant chest pain  - coughing up blood  - shortness of breath, wheezing, or hoarseness  - repeated problems with pneumonia or bronchitis  - swelling of the neck and face  - loss of appetite or weight loss  - fatigue. a cough that doesn't go away and gets worse over time constant chest pain coughing up blood shortness of breath, wheezing, or hoarseness repeated problems with pneumonia or bronchitis swelling of the neck and face loss of appetite or weight loss fatigue.

Common Signs and Symptoms When lung cancer first develops, there may be no symptoms at all. But if the cancer grows, it can cause changes that people should watch for. Common signs and symptoms of lung cancer include: - a cough that doesn't go away and 

In [50]:
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.1,
    do_sample=False,
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a medical assistant.

Answer ONLY using the supplied context.

If the answer is not present in the context,
say:

'I could not find sufficient evidence.'

Context:

The possible signs of lung cancer are: - a cough that doesn't go away and gets worse over time  - constant chest pain  - coughing up blood  - shortness of breath, wheezing, or hoarseness  - repeated problems with pneumonia or bronchitis  - swelling of the neck and face  - loss of appetite or weight loss  - fatigue. a cough that doesn't go away and gets worse over time constant chest pain coughing up blood shortness of breath, wheezing, or hoarseness repeated problems with pneumonia or bronchitis swelling of the neck and face loss of appetite or weight loss fatigue.

Common Signs and Symptoms When lung cancer first develops, there may be no symptoms at all. But if the cancer grows, it can cause changes that people should watch for. Common signs and symptoms of lung cancer include: - a cough that doesn't go away and 

In [77]:
import re
import warnings
import contextlib
import io

warnings.filterwarnings("ignore")

def ask_medical_llm(question):

    docs = retrieve_docs(
        question,
        top_k=50
    )

    docs = rerank_docs(
        question,
        docs,
        top_k=10
    )

    docs = quantum_rerank(
        docs,
        top_k=3
    )

    context = "\n\n".join(
        docs["Answer"].head(2).tolist()
    )

    prompt = f"""
You are a professional medical assistant.

Answer the user's question in 1-2 short paragraphs.

Rules:
- Do not repeat the context.
- Do not repeat the question.
- Keep the answer under 100 words.
- Use bullet points if needed.
- Give a concise explanation.

Context:
{context}

Question:
{question}

Final Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
    ).to("cuda")

    # Hide warning outputs
    f = io.StringIO()

    with contextlib.redirect_stderr(f):
        with contextlib.redirect_stdout(f):

            outputs = model.generate(
                **inputs,
                max_new_tokens=120,
                temperature=0.2,
                do_sample=False,
                repetition_penalty=1.15,
                pad_token_id=tokenizer.eos_token_id,
            )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    response = response.replace(
        "Final Answer:",
        ""
    ).strip()

    response = re.sub(
        r"\n{3,}",
        "\n\n",
        response
    )

    return response

In [78]:
import warnings
from transformers.utils import logging

logging.set_verbosity_error()

warnings.filterwarnings("ignore")

In [79]:
while True:

    q = input("\nAsk Medical Question: ")

    if q.lower() == "exit":
        break

    answer = ask_medical_llm(q)

    print("\n🩺 Medical Answer:\n")
    print(answer)


Ask Medical Question:  who is at risk for diabetes?



🩺 Medical Answer:

People at risk for Type 2 Diabetes include those:

- Overweight or obese adults aged 45+
- With a family history of diabetes
- Of certain ethnic backgrounds: African American, Alaskan Native, American Indian, Asian American, Hispanic/Latino, Pacific Islander 
- Those with a personal history of gestational diabetes, pre-pregnancy BMI > 9 lbs, or PCOS
- Adults with elevated BP (>140/90) or abnormal cholesterol/triglycerides
- Less physically active individuals  
- People with untreated sleep disorders like sleep apnea



Ask Medical Question:  okay What are symptoms of lung cancer 



🩺 Medical Answer:

Symptoms include:

• A persistent cough that worsens over time
• Constant chest pain 
• Coughing up blood (hemoptysis)
• Shortness of breath, wheezing, or hoarseness
• Frequent bouts of pneumonia or bronchitis
• Swelling around the neck and face
• Loss of appetite/weight loss
• Persistent tiredness/fatigue

These signs may indicate potential lung cancer but require further evaluation by healthcare professionals. • Persistent cough worsening over time • Chest pain constantly • Blood in sputum • Difficulty breathing, wheezing,



Ask Medical Question:  Okay Thank you 



🩺 Medical Answer:

• Reduce risks rather than eliminate them; focus on key areas like phones (emergency), kits (first aid), plans (family), alarms (smoke & CO), storage (guns). • For young kids: supervise closely and child-proof the environment. 
• Always follow instructions while handling tools/equipment safely. 
• Regular maintenance of installed devices ensures effectiveness. 
• This approach minimizes significant dangers without perfect safety. Key steps include immediate response readiness and physical barriers.


KeyboardInterrupt: Interrupted by user